# Configuration and test of floweaver Sankey(s) for IEDC dataset 1_F_Sand_Gravel_MFA_Global_184Countries_ZHUANG_2025

In [1]:
import pandas as pd
flows = pd.read_excel('1_F_Sand_Gravel_MFA_Global_184Countries_ZHUANG_2025.xlsx','Data') # Insert filename of IEDC download
flows

,id,dataset_name,aspect 1 : commodity,aspect 2 : material,aspect 3 : layer,aspect 4 : region,aspect 5 : time,aspect 6 : origin_process,aspect 7 : destination_process,aspect 8 : scenario,...,unitcode,unitcode1,stats_array_1,stats_array_2,stats_array_3,stats_array_4,comment,reserve1,reserve2,reserve3
0,3580974,1_F_Sand_Gravel_MFA_Global_184Countries_ZHUANG...,sand,sand and gravel,mass,Global,2019,lithosphere,market for sand,History,...,Pg,yr,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3580975,1_F_Sand_Gravel_MFA_Global_184Countries_ZHUANG...,sand,sand and gravel,mass,Global,2019,market for sand,market for concrete,History,...,Pg,yr,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3580976,1_F_Sand_Gravel_MFA_Global_184Countries_ZHUANG...,sand,sand and gravel,mass,Global,2019,market for sand,waste management,History,...,Pg,yr,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3580977,1_F_Sand_Gravel_MFA_Global_184Countries_ZHUANG...,"sand, in concrete",sand and gravel,mass,Global,2019,market for concrete,use phase - residential buildings,History,...,Pg,yr,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3580978,1_F_Sand_Gravel_MFA_Global_184Countries_ZHUANG...,"sand, in concrete",sand and gravel,mass,Global,2019,market for concrete,use phase - nonresidential buildings,History,...,Pg,yr,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
297,3581271,1_F_Sand_Gravel_MFA_Global_184Countries_ZHUANG...,"gravel, recycled or recovered",sand and gravel,mass,Global,1970,waste management,use phase - roads,History,...,Pg,yr,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
298,3581272,1_F_Sand_Gravel_MFA_Global_184Countries_ZHUANG...,gravel,sand and gravel,mass,Global,1970,market for gravel,use phase - residential buildings,History,...,Pg,yr,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
299,3581273,1_F_Sand_Gravel_MFA_Global_184Countries_ZHUANG...,gravel,sand and gravel,mass,Global,1970,market for gravel,use phase - nonresidential buildings,History,...,Pg,yr,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
300,3581274,1_F_Sand_Gravel_MFA_Global_184Countries_ZHUANG...,gravel,sand and gravel,mass,Global,1970,market for gravel,use phase - other construction,History,...,Pg,yr,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
import pandas as pd
flows       = pd.read_excel('1_F_Sand_Gravel_MFA_Global_184Countries_ZHUANG_2025.xlsx','Data')
material    = 'sand and gravel'
layer       = 'mass'
region      = 'Global'
scenario    = 'History'
time        =  2019

ExtractData = flows.loc[(flows['aspect 2 : material'] == material) & (flows['aspect 3 : layer'] == layer) & (flows['aspect 4 : region'] == region) & (flows['aspect 8 : scenario'] == scenario) & (flows['aspect 5 : time'] == time)]
SankeyDatar = ExtractData.drop(columns=['id','dataset_name','aspect 2 : material','aspect 3 : layer','aspect 4 : region','aspect 8 : scenario','aspect 5 : time','aspect 9 : ','aspect 10 : ','aspect 11 : ','aspect 12 : ','unitcode','unitcode1','stats_array_1','stats_array_2','stats_array_3','stats_array_4','comment','reserve1','reserve2','reserve3'])
SankeyDatar.rename(columns={'aspect 6 : origin_process': 'source', 'aspect 7 : destination_process': 'target','aspect 1 : commodity': 'type'}, inplace=True)
column_names= ['source','target','type','value']
SankeyDatar = SankeyDatar.reindex(columns=column_names)
SankeyDatar.reset_index(drop=True, inplace=True)
SankeyDatar

,source,target,type,value
0,lithosphere,market for sand,sand,10.327133
1,market for sand,market for concrete,sand,9.650642
2,market for sand,waste management,sand,0.420196
3,market for concrete,use phase - residential buildings,"sand, in concrete",3.426915
4,market for concrete,use phase - nonresidential buildings,"sand, in concrete",2.835457
5,market for concrete,use phase - other construction,"sand, in concrete",3.243510
6,market for concrete,waste management,"sand, in concrete",0.144760
7,use phase - residential buildings,waste management,"sand, in concrete",0.505463
8,use phase - nonresidential buildings,waste management,"sand, in concrete",0.582880
9,use phase - other construction,waste management,"sand, in concrete",0.712461


In [3]:
from floweaver import *

Sankey_version = 'Sand_Gravel_Global_ZHUANG_2025_Sankey_as_in_Paper'

# Set the default size to fit the documentation better.
size = dict(width=900, height=400)
SCALE = 6.0

nodes = {
    'sources':                ProcessGroup(['lithosphere']),
    'raw material markets':   ProcessGroup(['market for sand','market for gravel']),
    'main material markets':  ProcessGroup(['market for concrete','market for asphalt']),
    'use phase':              ProcessGroup(['use phase - residential buildings','use phase - nonresidential buildings',
                                'use phase - other construction','use phase - railways','use phase - roads']),
    'waste management':       ProcessGroup(['waste management']),    
    'finalsink':              ProcessGroup(['landfilling',]),
    'reused aggregates'        : Waypoint(direction='L'),
    'recycled aggregates'      : Waypoint(direction='L')}


nodes['raw material markets'].partition = Partition.Simple('process', ['market for sand','market for gravel'])
nodes['main material markets'].partition = Partition.Simple('process', ['market for concrete','market for asphalt'])
nodes['use phase'].partition = Partition.Simple('process', ['use phase - residential buildings','use phase - nonresidential buildings',
                                'use phase - other construction','use phase - railways','use phase - roads'])

ordering = [
    ['sources'],     
    ['raw material markets'],
    ['main material markets'],
    ['use phase','reused aggregates','recycled aggregates'],
    ['waste management'],
    ['finalsink'],  
]


bundles = [
    Bundle('sources',    'raw material markets'),
    Bundle('raw material markets', 'main material markets'),
    Bundle('raw material markets',  'use phase'),        
    Bundle('main material markets',  'use phase'),    
    Bundle('use phase',  'waste management'),      
    Bundle('raw material markets', 'waste management'),    
    Bundle('main material markets',  'waste management'),  
    Bundle('waste management',  'finalsink'),
    Bundle('waste management', 'use phase', waypoints=['reused aggregates']),
    Bundle('waste management', 'raw material markets', waypoints=['recycled aggregates'])]


# Partition by the "type/commodity" column of the flows table
flows_by_type = Partition.Simple('type', ['sand','sand, in concrete','sand, in asphalt','sand, recycled or recovered','sand, waste','gravel',
                                         'gravel, in concrete','gravel, in asphalt','gravel, recycled or recovered','gravel, waste'])


# Set the colours for the labels in the partition.
palette = {'sand':'sandybrown','sand in concrete':'sandybrown','sand in asphalt':'sandybrown','sand, recycled or recovered':'sandybrown',
           'sand, waste':'sandybrown','gravel':'slategrey','gravel in concrete':'slategrey','gravel in asphalt':'slategrey',
           'gravel, recycled or recovered':'slategrey','gravel, waste':'slategrey'}

# New SDD with the flow_partition set
sdd = SankeyDefinition(nodes, bundles, ordering, 
                       flow_partition=flows_by_type)

# w1 = weave(sdd, flows, palette=palette).to_widget(**size).auto_save_png('sand_example_v1.png')
Sankey = weave(sdd, SankeyDatar, palette=palette).to_widget(**size)
Sankey.scale = SCALE

#Sankey # Show below
Sankey.auto_save_png(Sankey_version + '_' + region + '_' + str(time) + '.png')
Sankey.auto_save_svg(Sankey_version + '_' + region + '_' + str(time) + '.svg')

In [4]:
# double-check and export Sankey specification as .json
if isinstance(SankeyDatar, pd.DataFrame):
    SankeyDatar = Dataset(SankeyDatar)
    
spec = compile_sankey_definition(
    sdd,       # see above: sdd = SankeyDefinition(nodes, bundles, ordering, flow_partition=flows_by_type)
    measures   = "value",
    link_width = None,
    link_color = None,
    palette    = palette,
    add_elsewhere_waypoints = True,
    dimension_tables        = None,
)

Sankey = execute_weave(spec, SankeyDatar)
Sankey.to_widget(**size)

In [5]:
# export Sankey spec as .json for use in IEDC web application
import json
json_data = json.dumps(spec.to_json(), indent=2)
# Write Sankey spec to .json, filename as indicated in: 
# https://github.com/IndEcol/IEDC_floweaver_integration/blob/main/floweaver_IEDC_Sankey_configs/floweaver_IEDC_Sankey_configs.config
with open(Sankey_version + '.json', "w", encoding="utf-8") as f:
    f.write(json_data)
    f.write("\n")